<a href="https://colab.research.google.com/github/asenin592/Programacion-de-Sistemas-de-Base-1/blob/main/Unidad%201/Collb/Gonzalez%20Francisco_PSSB_I_Colab_U1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Programación de Sistemas de Base I - Inspección de la Estructura de un Compilador y Procesamiento de Lenguajes
**Asignatura:** Programación de Sistemas de Base I (8.º Semestre)  
**Unidad I:** Introducción a la Compilación  
**Tiempo estimado:** 3 horas (Asíncrono / Guiado)  
**Repositorio Base / Entrega:** Integración con Moodle / GitHub

---


## 1. Objetivos de Aprendizaje
1. Identificar de forma tangible la diferencia entre ejecución compilada e interpretada utilizando las herramientas nativas del entorno (`gcc`, `python3`, `dis`, `ast`).
2. Explorar y visualizar la fase de Análisis Léxico y Sintáctico mediante la inspección del Árbol de Sintaxis Abstracta (AST) y la generación de Bytecode.
3. Comprender el rol de la Tabla de Símbolos en el seguimiento de identificadores durante el proceso de traducción.

---


## 2. Conexión Teórica (NotebookLM)
**Antes de continuar:** Accede al espacio de **NotebookLM del Curso** y realiza las siguientes preguntas de verificación:

- *¿Cuáles son las fases de la etapa de Análisis (Front-End) de un compilador?*  
**RESPUESTA:**   

Análisis lexicográfico (análisis léxico o morfológico): Es la primera fase del compilador. Su función consiste en leer el flujo de caracteres del código fuente de izquierda a derecha para identificar los límites de las palabras y agruparlos en unidades lógicas llamadas componentes léxicos o tokens (tales como identificadores de usuario, palabras reservadas, constantes numéricas u operadores). En esta fase también se realizan tareas auxiliares como la eliminación de comentarios y espacios en blanco innecesarios.

Análisis sintáctico (o parsing): Toma la secuencia de tokens proporcionada por el analizador léxico y comprueba si están estructurados y ordenados correctamente según las reglas gramaticales del lenguaje que se compila
. Conceptual u operativamente, su salida suele representarse mediante una estructura jerárquica en forma de árbol sintáctico.

Análisis semántico: Revisa el árbol sintáctico y la información contenida en la tabla de símbolos para evaluar si el programa posee cohesión, un significado correcto y respeta las directrices del lenguaje
. Su componente de mayor relevancia es la comprobación (o verificación) de tipos, la cual asegura que cada operador tenga operandos compatibles según la especificación del lenguaje (por ejemplo, validando la existencia de las variables, controlando que no se intente multiplicar un entero por una estructura de caracteres o que los parámetros de un subprograma coincidan con su declaración).    

Generación de código intermedio: Aunque actúa como transición hacia la etapa de síntesis, usualmente se agrupa dentro de las fases finales del Front-End. Esta fase traduce el programa fuente a un código simplificado diseñado para una máquina abstracta (como el código de tres direcciones o cuádruplas). Esto facilita la aplicación posterior de optimizaciones independientes de la plataforma antes de generar el código destino final.

Durante todo este flujo de análisis, el Front-End interactúa de manera constante con dos componentes transversales:

Administrador de la tabla de símbolos: Estructura de datos encargada de registrar los identificadores de usuario declarados en el programa y reunir información detallada sobre sus atributos (tales como su tipo de dato, ámbito o dirección en memoria).  

Manejador (o recuperación) de errores: Detecta, reporta y administra de manera informativa los errores sintácticos, léxicos o semánticos cometidos por el programador. Su propósito es lograr que el compilador se recupere del fallo y pueda continuar analizando el resto del código sin detenerse inmediatamente, generando así un informe completo de errores al final del proceso

- *¿Qué diferencia a un Lexema de un Token?*  
**RESPUESTA:** La diferencia fundamental entre un lexema y un token radica en que uno representa el texto real y concreto escrito en el código fuente, mientras que el otro representa la categoría lógica y abstracta que el compilador utiliza para analizar la estructura del programa.
Las diferencias específicas se detallan a continuación:
1. Lexema (La cadena concreta)
•	Definición: Es la secuencia real y física de caracteres en el programa fuente que coincide con un patrón determinado.
•	Nivel de abstracción: Es de bajo nivel; es el "texto plano" tal y como lo escribió el programador.
•	Ejemplos: miVariable, 105, 3.14, while o +.
2. Token (La categoría abstracta)
•	Definición: Es la categoría léxica abstracta asociada a un patrón. Funciona como el símbolo terminal que procesa y entiende el analizador sintáctico en su gramática.
•	Representación: Internamente, el compilador suele representarlo mediante un código numérico entero único (por ejemplo, 256 para identificadores) o como una tupla o par de la forma 〈nombre-token, valor-atributo〉. El "valor-atributo" suele ser un puntero a la tabla de símbolos para almacenar información detallada sobre el lexema.
•	Ejemplos: ID (para identificadores), NUM o NUM_ENT (para números enteros), ASIG (para asignación).  
La analogía para entender la diferencia
Como explican las fuentes, una excelente manera de comprender la distinción es la siguiente:    
•	El Token es como la palabra genérica "fruta".
•	El Lexema es la fruta concreta en sí misma, como "manzana" o "plátano".  
Relación entre Lexemas y Tokens
•	Relación uno a varios: Un solo token genérico puede representar un número infinito de lexemas diferentes. Por ejemplo, el token ID representa a los lexemas comision, fijo, valor, x o cualquier otra variable declarada por el usuario.  
•	Relación uno a uno: Hay casos especiales donde un token solo se asocia a un único lexema específico. Esto ocurre típicamente con palabras reservadas; por ejemplo, el token para la palabra clave while (o IF) tiene una correspondencia única y directa con el lexema escrito "while" (o "if").


- *¿Qué función cumple la Tabla de Símbolos durante la compilación?*  
**RESPUESTA:**  
La Tabla de Símbolos (también conocida como tabla de nombres o de identificadores) es una estructura de datos de alto rendimiento que funciona como un "diccionario de datos" para asistir al compilador en todas las fases del proceso de traducción. Su función principal consiste en registrar todos los identificadores de usuario (como nombres de variables, constantes, tipos, funciones o procedimientos) y almacenar y organizar sus atributos para tenerlos disponibles de manera rápida y eficiente.
A lo largo del flujo del compilador, la Tabla de Símbolos es un componente transversal que cumple las siguientes funciones fundamentales:
1. Control de Ámbito (Scope) y Visibilidad
En lenguajes estructurados en bloques, permite gestionar de manera precisa la visibilidad de los identificadores. Si una misma variable se declara con el mismo nombre en diferentes funciones o bloques anidados, la tabla de símbolos (frecuentemente estructurada como una pila de tablas hash o entornos encadenados) se encarga de determinar qué variable es la activa o visible en un punto específico del programa, aplicando la regla del bloque anidado más cercano para ocultar temporalmente las declaraciones más externas.
2. Soporte para el Análisis Semántico (Chequeo de Tipos y Restricciones)
El analizador semántico es el que más interactúa con la tabla para realizar comprobaciones sensibles al contexto y garantizar la cohesión del programa. Esto abarca:
•	Verificación de tipos: Comprobar que los operadores tengan operandos compatibles según la especificación del lenguaje (por ejemplo, validar que no se intente sumar una cadena con un entero).
•	Detección de variables no declaradas: Validar que todo identificador que se esté utilizando en el código de sentencias haya sido declarado previamente en la zona correspondiente.
•	Detección de redeclaraciones: Evitar que un identificador sea declarado más de una vez dentro del mismo ámbito.
•	Validación de llamadas a funciones: Asegurar que cuando se invoque un subprograma, los parámetros actuales coincidan en cantidad, tipo y orden con los parámetros formales definidos en la declaración de la función.
3. Generación de Código (Síntesis)
Durante la etapa de síntesis, el generador de código intermedio o final consulta de manera constante la Tabla de Símbolos para poder realizar la traducción hacia el lenguaje destino. Específicamente, extrae de ella:
•	Las direcciones de memoria (absolutas o relativas mediante desplazamientos o offsets) que representan físicamente a cada variable.
•	El tamaño en bytes u ocupación de memoria que requiere cada tipo o estructura de datos para reservar los espacios correspondientes en tiempo de ejecución.
•	Las constantes y variables temporales generadas automáticamente por el compilador.  
Tiempo de vida de la Tabla de Símbolos  
Por regla general, la tabla de símbolos permanece en memoria únicamente en tiempo de compilación. Una vez generado el archivo objeto o ejecutable final, esta estructura se descarta (a menos que se compile con opciones específicas de depuración). En el caso de los intérpretes, sin embargo, se mantiene en memoria durante toda la ejecución del programa ya que la traducción y la ejecución se producen de manera simultánea.




---

## 3. Sección Práctica 1: Pipeline de Compilación vs. Interpretación (CLI / Bash)
En esta sección analizaremos cómo el sistema operativo y el entorno procesan un lenguaje compilado (C) frente a uno interpretado/híbrido (Python).

### [CÓDIGO 1.1] Inspección de Herramientas del Sistema
```bash
!echo "=== COMPILADOR C DE LINUX (GCC) ==="
!gcc --version | head -n 1
!echo ""
!echo "=== INTÉRPRETE DE PYTHON ==="
!python3 --version

In [ ]:
!echo "=== COMPILADOR C DE LINUX (GCC) ==="
!gcc --version | head -n 1
!echo ""
!echo "=== INTÉRPRETE DE PYTHON ==="
!python3 --version

=== COMPILADOR C DE LINUX (GCC) ===
gcc (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0

=== INTÉRPRETE DE PYTHON ===
Python 3.13.15


### [CÓDIGO 1.2] Creación, Compilación y Análisis de Binario en C
```c
%%writefile hola_compilador.c
#include <stdio.h>
#define inputMsj "Escriba un numero entero: "
#define msj "La suma de %d + %d es: %d\n"

int sumar(int x, int y){
  return x + y;
}

int main() {
    int a = 5;
    int b;
    
    printf(inputMsj);
    scanf("%d",&b);
    int suma = sumar(a, b);
    printf(msj, a, b, suma);
    return 0;
}

In [ ]:
%%writefile hola_compilador.c
# include <stdio.h>
# define inputMsj "Escriba un numero entero: "
# define msj "La suma de %d + %d es: %d\n"

int sumar(int x, int y){
  return x + y;
}

int main() {
    int a = 5;
    int b;

    printf(inputMsj);
    scanf("%d",&b);
    int suma = sumar(a, b);
    printf(msj, a, b, suma);
    return 0;
}

Writing hola_compilador.c


## El Preprocesador (Antes del ensamblador)
Esta etapa expande las macros (como `#define`) e incluye los archivos de cabecera (como `#include <stdio.h>`).
```bash
!gcc -E hola_compilador.c -o hola_compilador.i
!echo === CÓDIGO PREPROCESADO ===
!cat hola_compilador.i | tail -n 20


In [ ]:
!gcc -E hola_compilador.c -o hola_compilador.i
!echo === CÓDIGO PREPROCESADO ===
!cat hola_compilador.i | tail -n 20

=== CÓDIGO PREPROCESADO ===
# 2 "hola_compilador.c" 2




# 5 "hola_compilador.c"
int sumar(int x, int y){
  return x + y;
}

int main() {
    int a = 5;
    int b;

    printf("Escriba un numero entero: ");
    scanf("%d",&b);
    int suma = sumar(a, b);
    printf("La suma de %d + %d es: %d\n", a, b, suma);
    return 0;
}


## Representación Intermedia (IR) y Optimizaciones
GCC utiliza una representación intermedia llamada GIMPLE antes de generar el ensamblador. Puedes ver cómo cambia el código antes y después de que el optimizador trabaje.

```bash
!gcc -O2 -fdump-tree-gimple hola_compilador.c -c
!echo === REPRESENTACIÓN INTERMEDIA (GIMPLE) ===
!cat hola_compilador.c.*gimple

In [ ]:
!gcc -O2 -fdump-tree-gimple hola_compilador.c -c
!echo === REPRESENTACIÓN INTERMEDIA (GIMPLE) ===
!cat hola_compilador.c.*gimple

hola_compilador.c: In function ‘main’:
hola_compilador.c:14:5: warning: ignoring return value of ‘scanf’ declared with attribute ‘warn_unused_result’ []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wunused-result-Wunused-result]8;;]
   14 |     scanf("%d",&b);
      |     ^~~~~~~~~~~~~~
/bin/bash: -c: line 1: syntax error near unexpected token `('
/bin/bash: -c: line 1: `echo === REPRESENTACIÓN INTERMEDIA (GIMPLE) ==='
int sumar (int x, int y)
{
  int D.2555;

  D.2555 = x + y;
  return D.2555;
}


int main ()
{
  int D.2557;

  {
    int a;
    int b;
    int suma;

    try
      {
        a = 5;
        printf ("Escriba un numero entero: ");
        scanf ("%d", &b);
        b.0_1 = b;
        suma = sumar (a, b.0_1);
        b.1_2 = b;
        printf ("La suma de %d + %d es: %d\n", a, b.1_2, suma);
        D.2557 = 0;
        return D.2557;
      }
    finally
      {
        b = {CLOBBER};
      }
  }
  D.2557 = 0;
  return D.2557;
}


__attribute__((artifici

## Compilación generando código ensamblador intermediate (.s)
 El compilador traduce el código **sin optimizaciones** línea por línea directamente a la memoria RAM (la pila o stack)

```bash
!gcc -S hola_compilador.c -o hola_compilador.s
!echo "=== CÓDIGO ENSAMBLADOR GENERADO (SÍNTESIS) ==="
!cat hola_compilador.s | head -n 25

In [ ]:
!gcc -S hola_compilador.c -o hola_compilador.s
!echo "=== CÓDIGO ENSAMBLADOR GENERADO (SÍNTESIS) ==="
!cat hola_compilador.s | head -n 25

=== CÓDIGO ENSAMBLADOR GENERADO (SÍNTESIS) ===
	.file	"hola_compilador.c"
	.text
	.globl	sumar
	.type	sumar, @function
sumar:
.LFB0:
	.cfi_startproc
	endbr64
	pushq	%rbp
	.cfi_def_cfa_offset 16
	.cfi_offset 6, -16
	movq	%rsp, %rbp
	.cfi_def_cfa_register 6
	movl	%edi, -4(%rbp)
	movl	%esi, -8(%rbp)
	movl	-4(%rbp), %edx
	movl	-8(%rbp), %eax
	addl	%edx, %eax
	popq	%rbp
	.cfi_def_cfa 7, 8
	ret
	.cfi_endproc
.LFE0:
	.size	sumar, .-sumar
	.section	.rodata


Para producir una traducción del código **optimizada** usa

```bash
!gcc -O3 -S hola_compilador.c -o hola_conmpilador_opt.s
!echo "=== CÓDIGO ENSAMBLADOR OPTIMIZADO GENERADO (SÍNTESIS) ==="
!cat hola_compilador_opt.s | head -n 25

In [ ]:
!gcc -O3 -S hola_compilador.c -o hola_compilador_opt.s
!echo "=== CÓDIGO ENSAMBLADOR OPTIMIZADO GENERADO (SÍNTESIS) ==="
!cat hola_compilador_opt.s | head -n 25

hola_compilador.c: In function ‘main’:
hola_compilador.c:14:5: warning: ignoring return value of ‘scanf’ declared with attribute ‘warn_unused_result’ []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wunused-result-Wunused-result]8;;]
   14 |     scanf("%d",&b);
      |     ^~~~~~~~~~~~~~
=== CÓDIGO ENSAMBLADOR OPTIMIZADO GENERADO (SÍNTESIS) ===
	.file	"hola_compilador.c"
	.text
	.p2align 4
	.globl	sumar
	.type	sumar, @function
sumar:
.LFB23:
	.cfi_startproc
	endbr64
	leal	(%rdi,%rsi), %eax
	ret
	.cfi_endproc
.LFE23:
	.size	sumar, .-sumar
	.section	.rodata.str1.1,"aMS",@progbits,1
.LC0:
	.string	"Escriba un numero entero: "
.LC1:
	.string	"%d"
.LC2:
	.string	"La suma de %d + %d es: %d\n"
	.section	.text.startup,"ax",@progbits
	.p2align 4
	.globl	main
	.type	main, @function


## Superando limitaciones de la arquitectura fija (X86_64)
La máquina virtual de Colab corre sobre procesadores Intel o AMD de 64 bits.  
**Impacto:** El código ensamblador generado (.s) siempre estará en sintaxis AT&T (por defecto en Linux) y para arquitectura x86_64. Para ver código ensamblador con sintaxis de Intel (más parecida a la de Windows), agregar el parámetro ***-masm=intel*** a la orden de GCC:

```bash
!gcc -S -masm=intel hola_compilador.c -o hola_compilador_8664.s
!echo "=== CÓDIGO ENSAMBLADOR x86_64 GENERADO (SÍNTESIS) ==="
!cat hola_compilador_8664.s | head -n 25


In [ ]:
!gcc -S -masm=intel hola_compilador.c -o hola_compilador_8664.s
!echo "=== CÓDIGO ENSAMBLADOR x86_64 GENERADO (SÍNTESIS) ==="
!cat hola_compilador_8664.s | head -n 25

=== CÓDIGO ENSAMBLADOR x86_64 GENERADO (SÍNTESIS) ===
	.file	"hola_compilador.c"
	.intel_syntax noprefix
	.text
	.globl	sumar
	.type	sumar, @function
sumar:
.LFB0:
	.cfi_startproc
	endbr64
	push	rbp
	.cfi_def_cfa_offset 16
	.cfi_offset 6, -16
	mov	rbp, rsp
	.cfi_def_cfa_register 6
	mov	DWORD PTR -4[rbp], edi
	mov	DWORD PTR -8[rbp], esi
	mov	edx, DWORD PTR -4[rbp]
	mov	eax, DWORD PTR -8[rbp]
	add	eax, edx
	pop	rbp
	.cfi_def_cfa 7, 8
	ret
	.cfi_endproc
.LFE0:
	.size	sumar, .-sumar


## Compilación a código objeto
```bash
# Generación del código objeto (antes de enlazar) y conteo líneas
!gcc -c hola_compilador.c -o hola_compilador.o
!objdump -d hola_compilador.o | wc -l

In [ ]:
# Generación del código objeto (antes de enlazar) y conteo líneas
!gcc -c hola_compilador.c -o hola_compilador.o
!objdump -d hola_compilador.o | wc -l

60


## Compilación a código máquina ejecutable

```bash
# Generación del código ejecutable final (después de enlazar) y conteo líneas
!gcc hola_compilador.c -o hola_compilador
!objdump -d hola_compilador | wc -l


In [ ]:
# Generación del código ejecutable final (después de enlazar) y conteo líneas
!gcc hola_compilador.c -o hola_compilador
!objdump -d hola_compilador | wc -l

194


## El "Monstruo Completo" (Enlace Estático)  
Por defecto, GCC usa ***enlace dinámico*** usando una estructura de datos tabular, la tabla **PLT** (*Procedure Linkage Table*) ya que el código usa `printf`, el programa necesita conectarse con la biblioteca estándar de C (`libc.so`). El enlazador no copia el código de `printf` dentro del archivo para no duplicar espacio en el disco; en su lugar, crea un "trampolín" o acceso directo en una sección llamada `<printf@plt>`.

Cada vez que se invoca a `printf`, realmente salta a este bloque plt, el cual averigua en qué parte de la memoria RAM del sistema operativo se encuentra la verdadera función `printf` y redirige el control allí en tiempo de ejecución.

Para ver cómo el enlazador inyecta literalmente miles y miles de líneas copiando físicamente el código de `printf` y todas sus dependencias dentro del ejecutable, se debe compilar de forma estática:

```bash
# Compilación estática
!gcc -static hola_compilador.c -o hola_estatico

# Conteo de líneas de código ensamblador
!objdump -d hola_estatico | wc -l


In [ ]:
# Compilación estática
!gcc -static hola_compilador.c -o hola_estatico

# Conteo de líneas de código ensamblador
!objdump -d hola_estatico | wc -l

182934


## Ejecución del código
```bash
!echo ""
!echo "=== EJECUCIÓN DEL PROGRAMA OBJETO ==="
!./hola_compilador


In [ ]:
!echo ""
!echo "=== EJECUCIÓN DEL PROGRAMA OBJETO ==="
!./hola_compilador


=== EJECUCIÓN DEL PROGRAMA OBJETO ===
Escriba un numero entero: 4
La suma de 5 + 4 es: 9


## 4. Sección Práctica 2: Revelando el Front-End (Análisis Léxico y AST)
Para comprender cómo el compilador "desarma" el código fuente en tokens y estructuras jerárquicas, inspeccionaremos el compilador de Python a nivel interno.

### [CÓDIGO 2.1] Inspección del Analizador Léxico (Tokenizador)

```python
import token
import tokenize
from io import BytesIO

# Código fuente de prueba en formato cadena
codigo_fuente = "suma = a + 10"

# Tokenización del código fuente
tokens = tokenize.tokenize(BytesIO(codigo_fuente.encode("utf-8")).readline)

print(
    f"{'LINEA/COL':<12} | {'TIPO DE TOKEN':<20} | {'VALOR (LEXEMA)':<15}"
)
print("-" * 55)
for tok in tokens:
    if tok.type in (
        tokenize.ENCODING,
        tokenize.ENDMARKER,
        tokenize.NL,
        tokenize.NEWLINE,
    ):
        continue
    nombre_token = token.tok_name[tok.type]
    posicion = f"{tok.start[0]}:{tok.start[1]}"
    print(f"{posicion:<12} | {nombre_token:<20} | {tok.string:<15}")


In [ ]:
import token
import tokenize
from io import BytesIO

# Código fuente de prueba en formato cadena
codigo_fuente = "suma = a + 10"

# Tokenización del código fuente
tokens = tokenize.tokenize(BytesIO(codigo_fuente.encode("utf-8")).readline)

print(
    f"{'LINEA/COL':<12} | {'TIPO DE TOKEN':<20} | {'VALOR (LEXEMA)':<15}"
)
print("-" * 55)
for tok in tokens:
    if tok.type in (
        tokenize.ENCODING,
        tokenize.ENDMARKER,
        tokenize.NL,
        tokenize.NEWLINE,
    ):
        continue
    nombre_token = token.tok_name[tok.type]
    posicion = f"{tok.start[0]}:{tok.start[1]}"
    print(f"{posicion:<12} | {nombre_token:<20} | {tok.string:<15}")

LINEA/COL    | TIPO DE TOKEN        | VALOR (LEXEMA) 
-------------------------------------------------------
1:0          | NAME                 | suma           
1:5          | OP                   | =              
1:7          | NAME                 | a              
1:9          | OP                   | +              
1:11         | NUMBER               | 10             


### [CÓDIGO 2.2] Construcción del Árbol de Sintaxis Abstracta (AST)

```python
import ast

# Generar y visualizar la estructura sintáctica
arbol = ast.parse("suma = a + 10")
print("=== ÁRBOLES DE SINTAXIS ABSTRACTA (REPRESENTACIÓN EN TEXTO) ===")
print(ast.dump(arbol, indent=4))


In [ ]:
import ast

# Generar y visualizar la estructura sintáctica
arbol = ast.parse("suma = a + 10")
print("=== ÁRBOLES DE SINTAXIS ABSTRACTA (REPRESENTACIÓN EN TEXTO) ===")
print(ast.dump(arbol, indent=4))

=== ÁRBOLES DE SINTAXIS ABSTRACTA (REPRESENTACIÓN EN TEXTO) ===
Module(
    body=[
        Assign(
            targets=[
                Name(id='suma', ctx=Store())],
            value=BinOp(
                left=Name(id='a', ctx=Load()),
                op=Add(),
                right=Constant(value=10)))])


## 5. Sección Práctica 3: Inspección de la Tabla de Símbolos y Bytecode
En esta sección simularemos la interacción con la Tabla de Símbolos y observaremos la generación de código intermedio.

### [CÓDIGO 3.1] Simulación de una Tabla de Símbolos básica en Python
```python
class TablaDeSimbolos:

    def __init__(self):
        self.simbolos = {}

    def insertar(self, nombre, tipo, valor=None, ambito="global"):
        if nombre in self.simbolos:
            print(f"[ERROR LÉXICO/SINTÁCTICO] Identificador '{nombre}' ya declarado.")
        else:
            self.simbolos[nombre] = {
                "tipo": tipo,
                "valor": valor,
                "ambito": ambito,
            }
            print(f"[TABLA DE SÍMBOLOS] Insertado: {nombre} ({tipo})")

    def buscar(self, nombre):
        return self.simbolos.get(nombre, None)

    def mostrar(self):
        print("\n=== CONTENIDO DE LA TABLA DE SÍMBOLOS ===")
        print(f"{'NOMBRE':<12} | {'TIPO':<10} | {'ÁMBITO':<10} | {'VALOR':<10}")
        print("-" * 50)
        for nombre, datos in self.simbolos.items():
            print(
                f"{nombre:<12} | {datos['tipo']:<10} | {datos['ambito']:<10} | {str(datos['valor']):<10}"
            )


# Prueba de la Tabla de Símbolos
ts = TablaDeSimbolos()
ts.insertar("a", "ENTERO", 5)
ts.insertar("b", "ENTERO", 10)
ts.insertar("suma", "ENTERO", 15)
ts.mostrar()
```


In [ ]:
class TablaDeSimbolos:

    def __init__(self):
        self.simbolos = {}

    def insertar(self, nombre, tipo, valor=None, ambito="global"):
        if nombre in self.simbolos:
            print(f"[ERROR LÉXICO/SINTÁCTICO] Identificador '{nombre}' ya declarado.")
        else:
            self.simbolos[nombre] = {
                "tipo": tipo,
                "valor": valor,
                "ambito": ambito,
            }
            print(f"[TABLA DE SÍMBOLOS] Insertado: {nombre} ({tipo})")

    def buscar(self, nombre):
        return self.simbolos.get(nombre, None)

    def mostrar(self):
        print("\n=== CONTENIDO DE LA TABLA DE SÍMBOLOS ===")
        print(f"{'NOMBRE':<12} | {'TIPO':<10} | {'ÁMBITO':<10} | {'VALOR':<10}")
        print("-" * 50)
        for nombre, datos in self.simbolos.items():
            print(
                f"{nombre:<12} | {datos['tipo']:<10} | {datos['ambito']:<10} | {str(datos['valor']):<10}"
            )


# Prueba de la Tabla de Símbolos
ts = TablaDeSimbolos()
ts.insertar("a", "ENTERO", 5)
ts.insertar("b", "ENTERO", 10)
ts.insertar("suma", "ENTERO", 15)
ts.mostrar()

[TABLA DE SÍMBOLOS] Insertado: a (ENTERO)
[TABLA DE SÍMBOLOS] Insertado: b (ENTERO)
[TABLA DE SÍMBOLOS] Insertado: suma (ENTERO)

=== CONTENIDO DE LA TABLA DE SÍMBOLOS ===
NOMBRE       | TIPO       | ÁMBITO     | VALOR     
--------------------------------------------------
a            | ENTERO     | global     | 5         
b            | ENTERO     | global     | 10        
suma         | ENTERO     | global     | 15        


### [CÓDIGO 3.2] Desensamblado a Código Intermedio / Bytecode (Máquina de Pila)
```python
import dis

def calcular():
    a = 5
    b = 10
    suma = a + b
    return suma


print("=== BYTECODE GENERADO PARA LA MÁQUINA VIRTUAL DE PYTHON ===")
dis.dis(calcular)
```

In [ ]:
import dis

def calcular():
    a = 5
    b = 10
    suma = a + b
    return suma


print("=== BYTECODE GENERADO PARA LA MÁQUINA VIRTUAL DE PYTHON ===")
dis.dis(calcular)

=== BYTECODE GENERADO PARA LA MÁQUINA VIRTUAL DE PYTHON ===
  3           RESUME                   0

  4           LOAD_CONST               1 (5)
              STORE_FAST               0 (a)

  5           LOAD_CONST               2 (10)
              STORE_FAST               1 (b)

  6           LOAD_FAST_LOAD_FAST      1 (a, b)
              BINARY_OP                0 (+)
              STORE_FAST               2 (suma)

  7           LOAD_FAST                2 (suma)
              RETURN_VALUE


## 6. Desafío Asíncrono / Entregable de la Unidad I (Avance del Producto Integrador)

### Contexto del Proyecto Integrador Autónomo:
Durante el semestre, cada equipo concebirá, diseñará e implementará un **Analizador Léxico y Sintáctico (Compiler Front-End)** para un lenguaje original propuesto por el propio equipo.

Ejemplos de proyectos de semestres anteriores:
- **DSL para Configuración de Robots / Drones:** Lenguaje de comandos simples (`FORWARD 10`, `ROTATE 90`).
- **Lenguaje de Consultas para Grafos/Tablas:** Alternativa simplificada a SQL (`SELECT age FROM users WHERE status == 1`).
- **Mini-Lenguaje Matemático / Scripting:** Soporte para vectores, matrices o evaluación de expresiones lógicas/aritméticas.
- **Lenguaje Formato/Marcado Personalizado:** Generador de reportes en HTML/Markdown a partir de una sintaxis limpia.

---

### Instrucciones del Desafío U1 (Fase 0: Definición e Inspección Inicial):

#### Parte A: Documento de Especificación del Lenguaje (RFC del Equipo)
Crea un archivo `ESPECIFICACION.docx` que contenga:
1. **Nombre del Lenguaje Original y Propósito:** ¿Qué problema resuelve o a quién va dirigido?
2. **Ejemplo de Código Fuente Válido:** Muestra un fragmento de código de al menos 10-15 líneas escrito en tu nuevo lenguaje.
3. **Catálogo Preliminar de Tokens:** Lista las palabras reservadas, identificadores, constantes (enteras, flotantes, cadenas) y operadores que usará tu lenguaje.

#### Parte B: Prototipo de Inspección en Código (Colab Execution)
Utilizando las celdas mágicas `%%writefile` (en Python o C):
1. Crea un archivo con una cadena que contenga tu ejemplo de código fuente original.
2. Implementa una función de prueba preliminar (o utiliza la librería `tokenize` de Python como simulador) que tome la cadena de tu lenguaje y la desglose en una lista de componentes léxicos.
3. Muestra una estructura en código que sirva como **Tabla de Símbolos inicial** donde se registren las variables declaradas en tu lenguaje.

---

### Criterios Mínimos que Debe Cumplir Cualquier Lenguaje Propuesto (Checklist de Viabilidad):
Para que la propuesta sea aprobada por el catedrático, el lenguaje debe cumplir con:
- [ ] Poseer al menos **3 tipos de tokens bien diferenciados** (ej. Palabras Reservadas, Identificadores, Literales).
- [ ] Incluir soporte para **expresiones aritméticas o lógicas** anidadas.
- [ ] Incluir al menos una **estructura de control de flujo** (ej. `if/else`, `while`, `repeat`) o una **estructura de bloques** (funciones, comandos).
- [ ] Definir un mecanismo explícito de asignación o declaración de datos.

---

### Autoevaluación Muestreada (Comprobación Tipo Gradiance)
Responde las siguientes preguntas analizando la sintaxis de tu nuevo lenguaje y valida tus razonamientos en NotebookLM:

1. **Pregunta 1 (Conflictos Léxicos):** Al revisar las palabras reservadas y los identificadores de tu lenguaje, ¿existe alguna regla léxica que pudiera causar ambigüedad (por ejemplo, que una palabra reservada coincida con el patrón de un identificador de usuario)? ¿Cómo la resolverá tu analizador?
2. **Pregunta 2 (Estructura de la Tabla de Símbolos):** De los elementos de tu lenguaje original, ¿cuáles atributos (tipo, valor, ámbito, dirección de memoria) necesitará almacenar tu Tabla de Símbolos cuando se procese una declaración?

---

### Formato de Entrega / Portafolio de Evidencias
1. Guarda este cuaderno con todas las salidas ejecutadas (`Archivo` -> `Guardar una copia en GitHub` / `PEREZ JUAN PSSB I Colab U1.ipynb`).
2. Guarda el documento `ESPECIFICACION.docx` con la especificación de tu lenguaje en tu repositorio personal (no repositorios grupales) de GitHub.
3. Registra en **Moodle** el enlace del cuaderno ejecutable en Colab y el de tu repositorio conteniendo la especificación formal del proyecto.

## Sección de trabajo no dirigido
Prototipo de Inspección en Código (Colab Execution)
Utilizando las celdas mágicas `%%writefile` (en Python o C):
1. Crea un archivo con una cadena que contenga tu ejemplo de código fuente original.

In [ ]:
%%writefile codigo_fuente.txt
// Lenguaje: RoboDSL v1.0
// Propósito: Control simple de un actuador

VAR velocidad = 50
VAR distancia = 100

ROBOT_START
  FORWARD(distancia)
  ROTATE(90)
  IF velocidad > 40 THEN
    SPEED(velocidad)
  ELSE
    SPEED(10)
  ENDIF
ROBOT_STOP

Writing codigo_fuente.txt


2. Implementa una función de prueba preliminar (o utiliza la librería `tokenize` de Python como simulador) que tome la cadena de tu lenguaje y la desglose en una lista de componentes léxicos.

In [ ]:
import re

def analizador_lexico(ruta_archivo):
    # Definición de patrones para los tokens de RoboDSL
    patrones_tokens = [
        ('KEYWORD', r'\b(VAR|IF|THEN|ELSE|ENDIF|ROBOT_START|ROBOT_STOP|FORWARD|ROTATE|SPEED)\b'),
        ('NUMBER',  r'\d+'),
        ('ID',      r'[a-zA-Z_][a-zA-Z0-9_]*'),
        ('OP',      r'[=\+\-\*/><]'),
        ('LPAREN',  r'\('),
        ('RPAREN',  r'\)'),
        ('COMMENT', r'//.*'),
        ('SKIP',    r'[ \t\n]+'),
        ('MISMATCH',r'.'),
    ]

    regex_completa = '|'.join(f'(?P<{name}>{pattern})' for name, pattern in patrones_tokens)

    with open(ruta_archivo, 'r') as f:
        codigo = f.read()

    print(f"{'TIPO':<15} | {'VALOR':<20}")
    print("-" * 38)

    for mo in re.finditer(regex_completa, codigo):
        tipo = mo.lastgroup
        valor = mo.group()

        if tipo == 'SKIP' or tipo == 'COMMENT':
            continue
        elif tipo == 'MISMATCH':
            print(f"[ERROR LÉXICO] Carácter inesperado: {valor}")
        else:
            print(f"{tipo:<15} | {valor:<20}")

# Ejecutar el analizador sobre el archivo generado anteriormente
analizador_lexico('codigo_fuente.txt')

TIPO            | VALOR               
--------------------------------------
OP              | /                   
OP              | /                   
ID              | Lenguaje            
[ERROR LÉXICO] Carácter inesperado: :
ID              | RoboDSL             
ID              | v1                  
[ERROR LÉXICO] Carácter inesperado: .
NUMBER          | 0                   
OP              | /                   
OP              | /                   
ID              | Prop                
[ERROR LÉXICO] Carácter inesperado: ó
ID              | sito                
[ERROR LÉXICO] Carácter inesperado: :
ID              | Control             
ID              | simple              
ID              | de                  
ID              | un                  
ID              | actuador            
KEYWORD         | VAR                 
ID              | velocidad           
OP              | =                   
NUMBER          | 50                  
KEYWORD         | VAR        

3. Muestra una estructura en código que sirva como **Tabla de Símbolos inicial** donde se registren las variables declaradas en tu lenguaje.

In [ ]:
class TablaSimbolosRoboDSL:
    def __init__(self):
        # Diccionario para almacenar: nombre -> {tipo, valor, ambito}
        self.simbolos = {}

    def insertar(self, nombre, tipo, valor, ambito="global"):
        self.simbolos[nombre] = {
            "tipo": tipo,
            "valor": valor,
            "ambito": ambito
        }

    def mostrar(self):
        print(f"=== TABLA DE SÍMBOLOS INICIAL (RoboDSL) ===")
        print(f"{'IDENTIFICADOR':<15} | {'TIPO':<10} | {'VALOR':<10} | {'ÁMBITO':<10}")
        print("-" * 55)
        for nombre, info in self.simbolos.items():
            print(f"{nombre:<15} | {info['tipo']:<10} | {str(info['valor']):<10} | {info['ambito']:<10}")

# Instanciar y registrar las variables del código de ejemplo
ts_robo = TablaSimbolosRoboDSL()
ts_robo.insertar("velocidad", "NUMBER", 50)
ts_robo.insertar("distancia", "NUMBER", 100)

# Mostrar el estado inicial de la tabla
ts_robo.mostrar()

=== TABLA DE SÍMBOLOS INICIAL (RoboDSL) ===
IDENTIFICADOR   | TIPO       | VALOR      | ÁMBITO    
-------------------------------------------------------
velocidad       | NUMBER     | 50         | global    
distancia       | NUMBER     | 100        | global    


1. **Pregunta 1 (Conflictos Léxicos):** Al revisar las palabras reservadas y los identificadores de tu lenguaje, ¿existe alguna regla léxica que pudiera causar ambigüedad (por ejemplo, que una palabra reservada coincida con el patrón de un identificador de usuario)? ¿Cómo la resolverá tu analizador?

2. **Pregunta 2 (Estructura de la Tabla de Símbolos):** De los elementos de tu lenguaje original, ¿cuáles atributos (tipo, valor, ámbito, dirección de memoria) necesitará almacenar tu Tabla de Símbolos cuando se procese una declaración?

**RESPUESTA:**  

**Respuesta 1:** La ambigüedad ocurre porque las palabras clave (como `VAR` o `IF`) también cumplen con el patrón de un identificador. La solución en el analizador léxico que implementamos es el **orden de prioridad**: al colocar el patrón de `KEYWORD` antes que el de `ID` en la lista de expresiones regulares, el motor de búsqueda siempre preferirá identificar una palabra reservada antes que un nombre de variable genérico.  

**Respuesta 2:** Para procesar una declaración en **RoboDSL**, la Tabla de Símbolos debe almacenar al menos los siguientes atributos:
1. **Tipo de Dato:** Necesario para asegurar que solo se realicen operaciones válidas (ej. no sumar un texto a una distancia).
2. **Valor:** Para mantener el estado actual de las variables durante la ejecución o simulación.
3. **Ámbito (Scope):** Crucial si el lenguaje crece para soportar funciones o bloques, permitiendo diferenciar entre variables locales y globales.
4. **Dirección de Memoria/Offset:** En una implementación real hacia código máquina o bytecode, se requiere para saber dónde leer o escribir el dato físicamente en el stack o heap.